<a href="https://colab.research.google.com/github/sumalya41/QFin.Colab/blob/main/manifold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import ipywidgets as widgets
from ipywidgets import interact

In [ ]:
pip install streamlit

In [ ]:
import streamlit as st
import numpy as np
import plotly.graph_objects as go

# 1. Page Configuration
st.set_page_config(layout="wide", page_title="Risk Manifold Visualizer")
st.title("Visualizing the Curvature of Risk via Tangent Spaces")
st.markdown("Move the sliders to shift your local position ($P$) on the risk manifold and see where linear models break down.")

# 2. Sidebar Controls
st.sidebar.header("Market Parameters")
p_x = st.sidebar.slider("Asset Volatility (X Position)", -2.0, 2.0, 0.0, 0.1)
p_y = st.sidebar.slider("Market Liquidity (Y Position)", -2.0, 2.0, 0.0, 0.1)
curvature_intensity = st.sidebar.slider("Market Stress (Curvature)", 0.5, 3.0, 1.5, 0.1)

# 3. Define the Risk Manifold Function: f(x, y)
# We use a non-linear function to represent a warped risk landscape
def risk_manifold(x, y, intensity):
    return np.sin(x * intensity) * np.cos(y * intensity) - 0.1 * (x**2 + y**2)

# Exact derivatives for the tangent plane calculation: z = z0 + f_x*(x-x0) + f_y*(y-y0)
def derivatives(x, y, intensity):
    f_x = intensity * np.cos(x * intensity) * np.cos(y * intensity) - 0.2 * x
    f_y = -intensity * np.sin(x * intensity) * np.sin(y * intensity) - 0.2 * y
    return f_x, f_y

# 4. Generate Surface Grid Data
x_range = np.linspace(-2.5, 2.5, 60)
y_range = np.linspace(-2.5, 2.5, 60)
X, Y = np.meshgrid(x_range, y_range)
Z = risk_manifold(X, Y, curvature_intensity)

# 5. Calculate Point P and its Tangent Plane
z_p = risk_manifold(p_x, p_y, curvature_intensity)
f_x, f_y = derivatives(p_x, p_y, curvature_intensity)
Z_tangent = z_p + f_x * (X - p_x) + f_y * (Y - p_y)

# 6. Plotly 3D Visualization
fig = go.Figure()

# Add the Manifold Surface (Colored by Risk/Height)
fig.add_trace(go.Surface(
    x=X, y=Y, z=Z,
    colorscale="RdYlBu_r",  # Red = High Risk, Blue = Low Risk
    opacity=0.85,
    showscale=True,
    colorbar=dict(title="Risk Level")
))

# Add the Flat Tangent Plane (Transparent gray/white)
fig.add_trace(go.Surface(
    x=X, y=Y, z=Z_tangent,
    colorscale=[[0, 'rgba(200,200,200,0.4)'], [1, 'rgba(200,200,200,0.4)']],
    showscale=False,
    opacity=0.5
))

# Add Point P (The current market state)
fig.add_trace(go.Scatter3d(
    x=[p_x], y=[p_y], z=[z_p],
    mode='markers',
    marker=dict(size=8, color='black', symbol='circle'),
    name='Current State (P)'
))

# Add a few white vertical projection lines to show "Model Error"
# We pick a 4x4 sample grid around P to show the error lines
proj_x, proj_y = np.meshgrid(np.linspace(p_x-0.8, p_x+0.8, 4), np.linspace(p_y-0.8, p_y+0.8, 4))
for px, py in zip(proj_x.flatten(), proj_y.flatten()):
    pz_manifold = risk_manifold(px, py, curvature_intensity)
    pz_tangent = z_p + f_x * (px - p_x) + f_y * (py - p_y)

    fig.add_trace(go.Scatter3d(
        x=[px, px], y=[py, py], z=[pz_manifold, pz_tangent],
        mode='lines',
        line=dict(color='white', width=3),
        showlegend=False
    ))

# Camera & Layout Adjustments
fig.update_layout(
    scene=dict(
        xaxis_title='Asset Volatility',
        yaxis_title='Market Liquidity',
        zaxis_title='Systemic Risk',
        aspectratio=dict(x=1, y=1, z=0.7)
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    height=700
)

# Render in Streamlit
st.plotly_chart(fig, use_container_width=True)

# 7. Concept Explainer Section
st.info(
    "**How to read this:** The flat plane represents a standard linear model. "
    "Notice that close to Point P, the white error lines are tiny—the linear model holds up. "
    "But as you look further away, especially in high-curvature 'market stress' zones, "
    "the gap between the plane and the manifold widens drastically. That gap is the unmeasured risk."
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import ipywidgets as widgets
from ipywidgets import interact

def plot_calculus_probability(mu, sigma, a, b):
    """
    Plots a continuous normal distribution PDF and shades the accumulated
    probability between points 'a' and 'b'.
    """
    # Fix bounds if user sets lower bound greater than upper bound
    if a > b:
        a, b = b, a

    # 1. LANGUAGE: Set up the sample space domain (x-axis)
    # We look at 4 standard deviations out from the mean
    x = np.linspace(mu - 4*sigma, mu + 4*sigma, 1000)

    # Define the PDF function f(x)
    pdf = norm.pdf(x, mu, sigma)

    # 2. ACCUMULATION: Define the specific interval to integrate over
    x_fill = np.linspace(a, b, 500)
    pdf_fill = norm.pdf(x_fill, mu, sigma)

    # Calculate the exact accumulated probability (the integral)
    # P(a <= X <= b) = F(b) - F(a)
    probability = norm.cdf(b, mu, sigma) - norm.cdf(a, mu, sigma)

    # 3. PLOTTING THE VISUALIZATION
    plt.figure(figsize=(12, 6))

    # Plot the full PDF curve
    plt.plot(x, pdf, label=f'PDF: $f(x)$', color='#1f77b4', lw=2.5)

    # Shade the area representing the definite integral
    plt.fill_between(x_fill, pdf_fill, color='#2ca02c', alpha=0.4,
                     label=f'Accumulated Prob:\n$\int_{{{a}}}^{{{b}}} f(x)dx = {probability:.4f}$')

    # 4. CHANGE / LANGUAGE: Mark key landmarks (Mean and Bounds)
    plt.axvline(mu, color='red', linestyle='--', alpha=0.7, label=f'Mean ($\mu$) = {mu}')
    plt.axvline(a, color='black', linestyle=':', alpha=0.5)
    plt.axvline(b, color='black', linestyle=':', alpha=0.5)

    # Customizing the graph aesthetics
    plt.title('Calculus-Based Probability: Accumulation Over an Interval', fontsize=14, fontweight='bold')
    plt.xlabel('Random Variable ($X$)', fontsize=12)
    plt.ylabel('Probability Density ($f(x)$)', fontsize=12)
    plt.xlim(mu - 4*sigma, mu + 4*sigma)
    plt.ylim(0, max(pdf) * 1.1)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper right', fontsize=11)

    plt.show()

# Create interactive sliders for every parameter
interact(
    plot_calculus_probability,
    mu=widgets.FloatSlider(value=0.0, min=-10.0, max=10.0, step=0.5, description='Mean (μ):'),
    sigma=widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description='Std Dev (σ):'),
    a=widgets.FloatSlider(value=-1.0, min=-15.0, max=15.0, step=0.1, description='Lower Bound (a):'),
    b=widgets.FloatSlider(value=1.0, min=-15.0, max=15.0, step=0.1, description='Upper Bound (b):')
);

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import ipywidgets as widgets
from ipywidgets import interact, interactive, HBox, VBox

def plot_distributions(dist_type, param1, param2, a, b):
    """
    Plots various continuous probability distributions based on user selection
    and accumulates probability between 'a' and 'b'.
    """
    plt.figure(figsize=(12, 6))

    # Ensure lower bound is less than upper bound
    if a > b:
        a, b = b, a

    # 1. SETUP DISTRIBUTION OBJECTS & DOMAINS BASED ON SELECTION
    if dist_type == 'Normal (Bell Curve)':
        mu, sigma = param1, param2
        dist = stats.norm(loc=mu, scale=sigma)
        x = np.linspace(mu - 4*sigma, mu + 4*sigma, 1000)
        title_str = f'Normal Distribution ($\mu={mu}$, $\sigma={sigma}$)'

    elif dist_type == 'Exponential (Decay)':
        lam = param1  # Rate parameter
        scale = 1 / lam
        dist = stats.expon(scale=scale)
        x = np.linspace(0, stats.expon.ppf(0.999, scale=scale), 1000)
        title_str = f'Exponential Distribution ($\lambda={lam}$)'

    elif dist_type == 'Uniform (Flat)':
        low, high = param1, param2
        if low >= high:
            high = low + 0.1 # Prevent error where low >= high
        dist = stats.uniform(loc=low, scale=high-low)
        x = np.linspace(low - 1, high + 1, 1000)
        title_str = f'Uniform Distribution (Interval: [{low}, {high}])'

    elif dist_type == 'Student\'s t (Heavy Tails)':
        df, scale = max(0.1, param1), param2
        dist = stats.t(df=df, loc=0, scale=scale)
        x = np.linspace(-4*scale, 4*scale, 1000)
        title_str = f"Student's t-Distribution ($df={df}$, $scale={scale}$)"

    # 2. CALCULATE PDF AND ACCUMULATION (INTEGRAL)
    pdf = dist.pdf(x)

    # Clip fill range to valid domain of the distribution
    x_fill = np.linspace(max(a, x[0]), min(b, x[-1]), 500)
    pdf_fill = dist.pdf(x_fill)

    # Integral evaluation using CDF: P(a <= X <= b) = F(b) - F(a)
    prob = dist.cdf(b) - dist.cdf(a)

    # 3. GRAPH VISUALIZATION
    plt.plot(x, pdf, label='Probability Density $f(x)$', color='#1f77b4', lw=2.5)
    plt.fill_between(x_fill, pdf_fill, color='#2ca02c', alpha=0.4,
                     label=f'Accumulated Area:\n$\int_{{{a}}}^{{{b}}} f(x)dx = {prob:.4f}$')

    # Aesthetics
    plt.title(f'Calculus-Based Probability: {title_str}', fontsize=14, fontweight='bold')
    plt.xlabel('Random Variable ($X$)', fontsize=12)
    plt.ylabel('Density ($f(x)$)', fontsize=12)
    plt.ylim(0, max(pdf) * 1.1)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper right', fontsize=11)
    plt.show()

# --- DYNAMIC WIDGET INTERACTION MATRIX ---
dist_dropdown = widgets.Dropdown(
    options=['Normal (Bell Curve)', 'Exponential (Decay)', 'Uniform (Flat)', "Student's t (Heavy Tails)"],
    value='Normal (Bell Curve)',
    description='Distribution:'
)

# Generic parameter sliders that re-purpose names dynamically
p1_slider = widgets.FloatSlider(value=0.0, min=-10.0, max=10.0, step=0.1, description='Param 1:')
p2_slider = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='Param 2:')
a_slider = widgets.FloatSlider(value=-1.0, min=-15.0, max=15.0, step=0.1, description='Lower Bound (a):')
b_slider = widgets.FloatSlider(value=1.0, min=-15.0, max=15.0, step=0.1, description='Upper Bound (b):')

def update_slider_labels(*args):
    """Dynamically updates slider functions and names based on chosen distribution."""
    if dist_dropdown.value == 'Normal (Bell Curve)':
        p1_slider.description = 'Mean (μ):'
        p1_slider.value, p1_slider.min, p1_slider.max = 0.0, -10.0, 10.0
        p2_slider.description = 'Std Dev (σ):'
        p2_slider.value, p2_slider.min, p2_slider.max = 1.0, 0.1, 5.0
        p2_slider.disabled = False
    elif dist_dropdown.value == 'Exponential (Decay)':
        p1_slider.description = 'Rate (λ):'
        p1_slider.value, p1_slider.min, p1_slider.max = 1.0, 0.1, 5.0
        p2_slider.description = 'N/A'
        p2_slider.disabled = True
    elif dist_dropdown.value == 'Uniform (Flat)':
        p1_slider.description = 'Left Bound:'
        p1_slider.value, p1_slider.min, p1_slider.max = 0.0, -5.0, 5.0
        p2_slider.description = 'Right Bound:'
        p2_slider.value, p2_slider.min, p2_slider.max = 5.0, 0.1, 10.0
        p2_slider.disabled = False
    elif dist_dropdown.value == "Student's t (Heavy Tails)":
        p1_slider.description = 'Deg Freedom(df):'
        p1_slider.value, p1_slider.min, p1_slider.max = 2.0, 1.0, 30.0
        p2_slider.description = 'Scale:'
        p2_slider.value, p2_slider.min, p2_slider.max = 1.0, 0.1, 5.0
        p2_slider.disabled = False

dist_dropdown.observe(update_slider_labels, 'value')
update_slider_labels() # Trigger baseline layout

# Bind everything together
interactive_plot = interactive(plot_distributions, dist_type=dist_dropdown, param1=p1_slider, param2=p2_slider, a=a_slider, b=b_slider)
output = interactive_plot.children[-1]
controls = VBox(interactive_plot.children[:-1])
display(controls, output)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive, VBox

def plot_functions(func_type, p1, p2, p3, p4):
    """
    Plots 7 different types of mathematical functions based on selection
    with dynamically mapped coefficients.
    """
    # Create a dense domain for smooth plotting
    x = np.linspace(-10, 10, 1000)
    y = np.zeros_like(x)
    title_str = ""

    # 1. POLYNOMIALS: f(x) = ax^3 + bx^2 + cx + d
    if func_type == '1. Polynomial (Cubic)':
        a, b, c, d = p1, p2, p3, p4
        y = a*x**3 + b*x**2 + c*x + d
        title_str = f'$f(x) = ({a})x^3 + ({b})x^2 + ({c})x + ({d})$'

    # 2. RATIONAL FUNCTIONS: f(x) = a / (bx - c) + d
    elif func_type == '2. Rational':
        a, b, c, d = p1, p2, p3, p4
        # Avoid division by zero by masking out the vertical asymptote
        with np.errstate(divide='ignore', invalid='ignore'):
            y = a / (b*x - c) + d
            # Mask extreme values near the vertical asymptote for cleaner plotting
            y[np.abs(y) > 100] = np.nan
        title_str = f'$f(x) = \ Monterrey / ({b}x - {c}) + {d}$'

    # 3. EXPONENTIAL FUNCTIONS: f(x) = a * b^(x - c) + d
    elif func_type == '3. Exponential':
        a, b, c, d = p1, p2, p3, p4
        # Base must be positive. Force b to be positive.
        b = max(0.01, b)
        y = a * (b**(x - c)) + d
        title_str = f'$f(x) = ({a}) \cdot ({b:.2f})^{{x - {c}}} + {d}$'

    # 4. LOGARITHMIC FUNCTIONS: f(x) = a * ln(bx - c) + d
    elif func_type == '4. Logarithmic':
        a, b, c, d = p1, p2, p3, p4
        # Log domain requirement: bx - c > 0
        with np.errstate(invalid='ignore', divide='ignore'):
            mask = (b*x - c) > 0
            y = np.where(mask, a * np.log(b*x - c) + d, np.nan)
        title_str = f'$f(x) = ({a})\ln({b}x - {c}) + {d}$'

    # 5. TRIGONOMETRIC FUNCTIONS: f(x) = a * sin(bx - c) + d
    elif func_type == '5. Trigonometric (Sin)':
        a, b, c, d = p1, p2, p3, p4
        y = a * np.sin(b*x - c) + d
        title_str = f'$f(x) = ({a})\sin({b}x - {c}) + {d}$'

    # 6. ABSOLUTE FUNCTIONS: f(x) = a * |bx - c| + d
    elif func_type == '6. Absolute Value':
        a, b, c, d = p1, p2, p3, p4
        y = a * np.abs(b*x - c) + d
        title_str = f'$f(x) = ({a})|{b}x - {c}| + {d}$'

    # 7. PIECEWISE FUNCTIONS: If x < c, f(x) = ax + b; Else, f(x) = sin(x) + d
    elif func_type == '7. Piecewise':
        a, b, c, d = p1, p2, p3, p4
        y = np.where(x < c, a*x + b, np.sin(x) + d)
        title_str = f'$f(x) = {a}x + {b}$ if $x < {c}$ else $\sin(x) + {d}$'

    # --- GRAPH GENERATION ---
    plt.figure(figsize=(12, 6))
    plt.plot(x, y, label=f'Selected Graph', color='#d62728', lw=2.5)

    # Structural guides
    plt.axhline(0, color='black', linewidth=1.2, alpha=0.5)
    plt.axvline(0, color='black', linewidth=1.2, alpha=0.5)

    # Extra visual additions for specific functions
    if func_type == '2. Rational' and b != 0:
        plt.axvline(c/b, color='blue', linestyle='--', alpha=0.5, label=f'Asymptote ($x = {c/b:.2f}$)')
    if func_type == '7. Piecewise':
        plt.axvline(c, color='purple', linestyle=':', alpha=0.7, label=f'Boundary ($x = {c}$)')

    # Styling and Limits
    plt.title(f'Dynamic Function Explorer: {title_str}', fontsize=14, fontweight='bold')
    plt.xlabel('$x$', fontsize=12)
    plt.ylabel('$f(x)$', fontsize=12)
    plt.xlim(-10, 10)
    plt.ylim(-10, 10)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper right', fontsize=11)
    plt.show()

# --- CONFIGURING INTERACTIVE WIDGET MATRIX ---
func_dropdown = widgets.Dropdown(
    options=[
        '1. Polynomial (Cubic)',
        '2. Rational',
        '3. Exponential',
        '4. Logarithmic',
        '5. Trigonometric (Sin)',
        '6. Absolute Value',
        '7. Piecewise'
    ],
    value='1. Polynomial (Cubic)',
    description='Function Type:'
)

# Standard parameter sliders that change behaviors seamlessly
slider1 = widgets.FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, description='a:')
slider2 = widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='b:')
slider3 = widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='c:')
slider4 = widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='d:')

def update_labels(*args):
    """Re-maps UI slider controls contextualizing parameters based on selections."""
    ft = func_dropdown.value
    if ft == '1. Polynomial (Cubic)':
        slider1.description, slider1.value = 'a ($x^3$ coef):', 0.5
        slider2.description, slider2.value = 'b ($x^2$ coef):', -0.2
        slider3.description, slider3.value = 'c ($x$ coef):', -2.0
        slider4.description, slider4.value = 'd (constant):', 1.0
    elif ft == '2. Rational':
        slider1.description, slider1.value = 'a (numerator):', 1.0
        slider2.description, slider2.value = 'b (x scaling):', 1.0
        slider3.description, slider3.value = 'c (horiz shift):', 0.0
        slider4.description, slider4.value = 'd (vert shift):', 0.0
    elif ft == '3. Exponential':
        slider1.description, slider1.value = 'a (vertical scale):', 1.0
        slider2.description, slider2.value = 'b (base > 0):', 2.0
        slider2.min, slider2.max = 0.01, 5.0  # Limit base bounds to avoid imaginary projections
        slider3.description, slider3.value = 'c (horiz shift):', 0.0
        slider4.description, slider4.value = 'd (vert shift):', 0.0
    elif ft == '4. Logarithmic':
        slider1.description, slider1.value = 'a (vertical scale):', 1.0
        slider2.description, slider2.value = 'b (x scaling):', 1.0
        slider3.description, slider3.value = 'c (horiz shift):', 0.0
        slider4.description, slider4.value = 'd (vert shift):', 0.0
    elif ft == '5. Trigonometric (Sin)':
        slider1.description, slider1.value = 'a (amplitude):', 2.0
        slider2.description, slider2.value = 'b (frequency):', 1.0
        slider3.description, slider3.value = 'c (phase shift):', 0.0
        slider4.description, slider4.value = 'd (vert shift):', 0.0
    elif ft == '6. Absolute Value':
        slider1.description, slider1.value = 'a (slope/stretch):', 1.0
        slider2.description, slider2.value = 'b (x scaling):', 1.0
        slider3.description, slider3.value = 'c (horiz shift):', 0.0
        slider4.description, slider4.value = 'd (vert shift):', 0.0
    elif ft == '7. Piecewise':
        slider1.description, slider1.value = 'a (linear slope):', 0.5
        slider2.description, slider2.value = 'b (linear intercept):', 2.0
        slider3.description, slider3.value = 'c (split point):', 0.0
        slider4.description, slider4.value = 'd (trig shift):', -1.0

func_dropdown.observe(update_labels, 'value')
update_labels() # Establish default parameters initially

# Connect sliders to live display window
interactive_plot = interactive(plot_functions, func_type=func_dropdown, p1=slider1, p2=slider2, p3=slider3, p4=slider4)
output = interactive_plot.children[-1]
controls = VBox(interactive_plot.children[:-1])
display(controls, output)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive, VBox

def plot_functions(func_type, p1, p2, p3, p4, trig_type='sin'):
    """
    Plots 7 different types of mathematical functions based on selection
    with dynamically mapped coefficients, now supporting multiple trig types.
    """
    # Create a dense domain for smooth plotting
    x = np.linspace(-10, 10, 1000)
    y = np.zeros_like(x)
    title_str = ""

    # 1. POLYNOMIALS: f(x) = ax^3 + bx^2 + cx + d
    if func_type == '1. Polynomial (Cubic)':
        a, b, c, d = p1, p2, p3, p4
        y = a*x**3 + b*x**2 + c*x + d
        title_str = f'$f(x) = ({a})x^3 + ({b})x^2 + ({c})x + ({d})$'

    # 2. RATIONAL FUNCTIONS: f(x) = a / (bx - c) + d
    elif func_type == '2. Rational':
        a, b, c, d = p1, p2, p3, p4
        with np.errstate(divide='ignore', invalid='ignore'):
            y = a / (b*x - c) + d
            y[np.abs(y) > 100] = np.nan
        title_str = f'$f(x) = {a} / ({b}x - {c}) + {d}$'

    # 3. EXPONENTIAL FUNCTIONS: f(x) = a * b^(x - c) + d
    elif func_type == '3. Exponential':
        a, b, c, d = p1, p2, p3, p4
        b = max(0.01, b)
        y = a * (b**(x - c)) + d
        title_str = f'$f(x) = ({a}) \cdot ({b:.2f})^{{x - {c}}} + {d}$'

    # 4. LOGARITHMIC FUNCTIONS: f(x) = a * ln(bx - c) + d
    elif func_type == '4. Logarithmic':
        a, b, c, d = p1, p2, p3, p4
        with np.errstate(invalid='ignore', divide='ignore'):
            mask = (b*x - c) > 0
            y = np.where(mask, a * np.log(b*x - c) + d, np.nan)
        title_str = f'$f(x) = ({a})\ln({b}x - {c}) + {d}$'

    # 5. TRIGONOMETRIC FUNCTIONS: sin, cos, or tan
    elif func_type == '5. Trigonometric':
        a, b, c, d = p1, p2, p3, p4

        if trig_type == 'sin':
            y = a * np.sin(b*x - c) + d
            title_str = f'$f(x) = ({a})\sin({b}x - {c}) + {d}$'
        elif trig_type == 'cos':
            y = a * np.cos(b*x - c) + d
            title_str = f'$f(x) = ({a})\cos({b}x - {c}) + {d}$'
        elif trig_type == 'tan':
            with np.errstate(divide='ignore', invalid='ignore'):
                y = a * np.tan(b*x - c) + d
                # Mask out the infinite vertical asymptotes unique to Tangent
                y[np.abs(y) > 20] = np.nan
            title_str = f'$f(x) = ({a})\\tan({b}x - {c}) + {d}$'

    # 6. ABSOLUTE FUNCTIONS: f(x) = a * |bx - c| + d
    elif func_type == '6. Absolute Value':
        a, b, c, d = p1, p2, p3, p4
        y = a * np.abs(b*x - c) + d
        title_str = f'$f(x) = ({a})|{b}x - {c}| + {d}$'

    # 7. PIECEWISE FUNCTIONS: Hybrid linear and trig split
    elif func_type == '7. Piecewise':
        a, b, c, d = p1, p2, p3, p4
        y = np.where(x < c, a*x + b, np.sin(x) + d)
        title_str = f'$f(x) = {a}x + {b}$ if $x < {c}$ else $\sin(x) + {d}$'

    # --- GRAPH GENERATION ---
    plt.figure(figsize=(12, 6))
    plt.plot(x, y, label=f'Selected Graph', color='#d62728', lw=2.5)

    # Structural guides
    plt.axhline(0, color='black', linewidth=1.2, alpha=0.5)
    plt.axvline(0, color='black', linewidth=1.2, alpha=0.5)

    if func_type == '2. Rational' and b != 0:
        plt.axvline(c/b, color='blue', linestyle='--', alpha=0.5, label=f'Asymptote ($x = {c/b:.2f}$)')
    if func_type == '7. Piecewise':
        plt.axvline(c, color='purple', linestyle=':', alpha=0.7, label=f'Boundary ($x = {c}$)')

    # Styling and Limits
    plt.title(f'Dynamic Function Explorer: {title_str}', fontsize=14, fontweight='bold')
    plt.xlabel('$x$', fontsize=12)
    plt.ylabel('$f(x)$', fontsize=12)
    plt.xlim(-10, 10)
    plt.ylim(-10, 10)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper right', fontsize=11)
    plt.show()

# --- CONFIGURING INTERACTIVE WIDGET MATRIX ---
func_dropdown = widgets.Dropdown(
    options=[
        '1. Polynomial (Cubic)',
        '2. Rational',
        '3. Exponential',
        '4. Logarithmic',
        '5. Trigonometric',
        '6. Absolute Value',
        '7. Piecewise'
    ],
    value='1. Polynomial (Cubic)',
    description='Function Type:'
)

# New dropdown explicitly for selecting the trig variant
trig_dropdown = widgets.Dropdown(
    options=['sin', 'cos', 'tan'],
    value='sin',
    description='Trig Subtype:',
    disabled=True # Hidden/Disabled until option 5 is selected
)

slider1 = widgets.FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, description='a:')
slider2 = widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='b:')
slider3 = widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='c:')
slider4 = widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='d:')

def update_labels(*args):
    """Re-maps UI slider controls contextualizing parameters based on selections."""
    ft = func_dropdown.value

    # Handle the visibility of the sub-trig dropdown menu
    if ft == '5. Trigonometric':
        trig_dropdown.disabled = False
    else:
        trig_dropdown.disabled = True

    if ft == '1. Polynomial (Cubic)':
        slider1.description, slider1.value = 'a ($x^3$ coef):', 0.5
        slider2.description, slider2.value = 'b ($x^2$ coef):', -0.2
        slider3.description, slider3.value = 'c ($x$ coef):', -2.0
        slider4.description, slider4.value = 'd (constant):', 1.0
    elif ft == '2. Rational':
        slider1.description, slider1.value = 'a (numerator):', 1.0
        slider2.description, slider2.value = 'b (x scaling):', 1.0
        slider3.description, slider3.value = 'c (horiz shift):', 0.0
        slider4.description, slider4.value = 'd (vert shift):', 0.0
    elif ft == '3. Exponential':
        slider1.description, slider1.value = 'a (vertical scale):', 1.0
        slider2.description, slider2.value = 'b (base > 0):', 2.0
        slider2.min, slider2.max = 0.01, 5.0
        slider3.description, slider3.value = 'c (horiz shift):', 0.0
        slider4.description, slider4.value = 'd (vert shift):', 0.0
    elif ft == '4. Logarithmic':
        slider1.description, slider1.value = 'a (vertical scale):', 1.0
        slider2.description, slider2.value = 'b (x scaling):', 1.0
        slider3.description, slider3.value = 'c (horiz shift):', 0.0
        slider4.description, slider4.value = 'd (vert shift):', 0.0
    elif ft == '5. Trigonometric':
        slider1.description, slider1.value = 'a (amplitude):', 2.0
        slider2.description, slider2.value = 'b (frequency):', 1.0
        slider3.description, slider3.value = 'c (phase shift):', 0.0
        slider4.description, slider4.value = 'd (vert shift):', 0.0
    elif ft == '6. Absolute Value':
        slider1.description, slider1.value = 'a (slope/stretch):', 1.0
        slider2.description, slider2.value = 'b (x scaling):', 1.0
        slider3.description, slider3.value = 'c (horiz shift):', 0.0
        slider4.description, slider4.value = 'd (vert shift):', 0.0
    elif ft == '7. Piecewise':
        slider1.description, slider1.value = 'a (linear slope):', 0.5
        slider2.description, slider2.value = 'b (linear intercept):', 2.0
        slider3.description, slider3.value = 'c (split point):', 0.0
        slider4.description, slider4.value = 'd (trig shift):', -1.0

func_dropdown.observe(update_labels, 'value')
update_labels()

# Connect dropdowns and sliders to live display window
interactive_plot = interactive(plot_functions, func_type=func_dropdown, trig_type=trig_dropdown, p1=slider1, p2=slider2, p3=slider3, p4=slider4)
output = interactive_plot.children[-1]
controls = VBox(interactive_plot.children[:-1])
display(controls, output)

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import multivariate_normal
import ipywidgets as widgets
from ipywidgets import interactive, VBox, HBox

def plot_multivariate_prob(mu_x, mu_y, sigma_x, sigma_y, rho, a, b, c, d):
    """
    Plots a 3D Bivariate Normal Distribution surface and highlights
    the volume (accumulated probability) within a 2D boundary box.
    """
    # Fix bounds if lower is greater than upper
    if a > b: a, b = b, a
    if c > d: c, d = d, c

    # 1. THE LANGUAGE: Setup 3D space domain based on means and standard deviations
    x_min, x_max = mu_x - 3*sigma_x, mu_x + 3*sigma_x
    y_min, y_max = mu_y - 3*sigma_y, mu_y + 3*sigma_y

    x = np.linspace(x_min, x_max, 100)
    y = np.linspace(y_min, y_max, 100)
    X, Y = np.meshgrid(x, y)
    pos = np.dstack((X, Y))

    # Define Covariance Matrix using Standard Deviations and Correlation (rho)
    # Covariance = rho * sigma_x * sigma_y
    covariance = rho * sigma_x * sigma_y
    rv = multivariate_normal([mu_x, mu_y], [[sigma_x**2, covariance], [covariance, sigma_y**2]])
    Z = rv.pdf(pos)

    # 2. THE ACCUMULATION: Calculate exact probability in the [a,b] x [c,d] region
    # P(a <= X <= b, c <= Y <= d)
    prob = rv.cdf([b, d]) - rv.cdf([a, d]) - rv.cdf([b, c]) + rv.cdf([a, c])

    # Filter out the specific sub-grid to shade the accumulated volume region
    x_fill = np.linspace(max(a, x_min), min(b, x_max), 40)
    y_fill = np.linspace(max(c, y_min), min(d, y_max), 40)
    X_fill, Y_fill = np.meshgrid(x_fill, y_fill)
    pos_fill = np.dstack((X_fill, Y_fill))
    Z_fill = rv.pdf(pos_fill)

    # 3. GRAPHING WITH PLOTLY (Allows live 3D rotation)
    fig = go.Figure()

    # Add the main semi-transparent probability distribution surface
    fig.add_trace(go.Surface(
        x=x, y=y, z=Z,
        colorscale='Blues',
        opacity=0.6,
        showscale=False,
        name='Joint PDF f(x,y)'
    ))

    # Add the highlighted dense volume region representing the double integral
    fig.add_trace(go.Surface(
        x=x_fill, y=y_fill, z=Z_fill,
        colorscale='Greens',
        showscale=False,
        name='Accumulated Volume'
    ))

    # Update layout and titles
    fig.update_layout(
        title={
            'text': f"Multivariate Probability: P({a} ≤ X ≤ {b}, {c} ≤ Y ≤ {d}) = {prob:.4f}",
            'y':0.9, 'x':0.5, 'xanchor': 'center', 'yanchor': 'top',
            'font': dict(size=16, color='black')
        },
        scene=dict(
            xaxis=dict(title='Variable X', range=[x_min, x_max]),
            yaxis=dict(title='Variable Y', range=[y_min, y_max]),
            zaxis=dict(title='Density f(x,y)', range=[0, np.max(Z)*1.1]),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.2)) # Set an angled 3D viewpoint
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        width=800,
        height=600
    )

    fig.show()

# --- INTERACTIVE CONTROL INTERFACE ---
# Variables for X distribution
mu_x_s = widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.5, description='Mean X (μx):')
sigma_x_s = widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description='Std Dev X (σx):')

# Variables for Y distribution
mu_y_s = widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.5, description='Mean Y (μy):')
sigma_y_s = widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description='Std Dev Y (σy):')

# Correlation between X and Y (Controls the tilt/flow direction of the 3D hill)
rho_s = widgets.FloatSlider(value=0.0, min=-0.9, max=0.9, step=0.1, description='Correlation (ρ):')

# Definite Double Integral Boundary limits
a_s = widgets.FloatSlider(value=-1.0, min=-4.0, max=4.0, step=0.1, description='X Lower (a):')
b_s = widgets.FloatSlider(value=1.0, min=-4.0, max=4.0, step=0.1, description='X Upper (b):')
c_s = widgets.FloatSlider(value=-1.0, min=-4.0, max=4.0, step=0.1, description='Y Lower (c):')
d_s = widgets.FloatSlider(value=1.0, min=-4.0, max=4.0, step=0.1, description='Y Upper (d):')

# Pack UI into organized columns
col1 = VBox([mu_x_s, sigma_x_s, mu_y_s, sigma_y_s, rho_s])
col2 = VBox([a_s, b_s, c_s, d_s])
ui = HBox([col1, col2])

out = widgets.interactive_output(plot_multivariate_prob, {
    'mu_x': mu_x_s, 'mu_y': mu_y_s, 'sigma_x': sigma_x_s, 'sigma_y': sigma_y_s,
    'rho': rho_s, 'a': a_s, 'b': b_s, 'c': c_s, 'd': d_s
})

display(ui, out)

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from scipy.stats import multivariate_normal
import ipywidgets as widgets
from ipywidgets import interactive, VBox, HBox

# FORCE COLAB TO RENDER PLOTLY GRAPHICS SUCCESSFULLY
pio.renderers.default = 'colab'

def plot_multivariate_prob(mu_x, mu_y, sigma_x, sigma_y, rho, a, b, c, d):
    """
    Plots a 3D Bivariate Normal Distribution surface and highlights
    the volume (accumulated probability) within a 2D boundary box.
    """
    # Fix bounds if lower is greater than upper
    if a > b: a, b = b, a
    if c > d: c, d = d, c

    # 1. THE LANGUAGE: Setup 3D space domain
    x_min, x_max = mu_x - 3*sigma_x, mu_x + 3*sigma_x
    y_min, y_max = mu_y - 3*sigma_y, mu_y + 3*sigma_y

    x = np.linspace(x_min, x_max, 100)
    y = np.linspace(y_min, y_max, 100)
    X, Y = np.meshgrid(x, y)
    pos = np.dstack((X, Y))

    # Calculate covariance matrix
    covariance = rho * sigma_x * sigma_y
    rv = multivariate_normal([mu_x, mu_y], [[sigma_x**2, covariance], [covariance, sigma_y**2]])
    Z = rv.pdf(pos)

    # 2. THE ACCUMULATION: Calculate exact probability in the region
    prob = rv.cdf([b, d]) - rv.cdf([a, d]) - rv.cdf([b, c]) + rv.cdf([a, c])

    # Filter out sub-grid for shading volume
    x_fill = np.linspace(max(a, x_min), min(b, x_max), 40)
    y_fill = np.linspace(max(c, y_min), min(d, y_max), 40)
    X_fill, Y_fill = np.meshgrid(x_fill, y_fill)
    pos_fill = np.dstack((X_fill, Y_fill))
    Z_fill = rv.pdf(pos_fill)

    # 3. GRAPHING WITH PLOTLY
    fig = go.Figure()

    # Add main surface
    fig.add_trace(go.Surface(
        x=x, y=y, z=Z,
        colorscale='Blues',
        opacity=0.6,
        showscale=False,
        name='Joint PDF f(x,y)'
    ))

    # Add highlighted volume
    fig.add_trace(go.Surface(
        x=x_fill, y=y_fill, z=Z_fill,
        colorscale='Greens',
        showscale=False,
        name='Accumulated Volume'
    ))

    # Update layout
    fig.update_layout(
        title={
            'text': f"Multivariate Probability: P({a} ≤ X ≤ {b}, {c} ≤ Y ≤ {d}) = {prob:.4f}",
            'y':0.95, 'x':0.5, 'xanchor': 'center', 'yanchor': 'top',
            'font': dict(size=14, color='black')
        },
        scene=dict(
            xaxis=dict(title='Variable X', range=[x_min, x_max]),
            yaxis=dict(title='Variable Y', range=[y_min, y_max]),
            zaxis=dict(title='Density f(x,y)', range=[0, np.max(Z)*1.1]),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        width=750,
        height=550
    )

    fig.show()

# --- INTERACTIVE CONTROL INTERFACE ---
mu_x_s = widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.5, description='Mean X (μx):')
sigma_x_s = widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description='Std Dev X (σx):')
mu_y_s = widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.5, description='Mean Y (μy):')
sigma_y_s = widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description='Std Dev Y (σy):')
rho_s = widgets.FloatSlider(value=0.3, min=-0.9, max=0.9, step=0.1, description='Correlation (ρ):')

a_s = widgets.FloatSlider(value=-1.0, min=-4.0, max=4.0, step=0.1, description='X Lower (a):')
b_s = widgets.FloatSlider(value=1.0, min=-4.0, max=4.0, step=0.1, description='X Upper (b):')
c_s = widgets.FloatSlider(value=-1.0, min=-4.0, max=4.0, step=0.1, description='Y Lower (c):')
d_s = widgets.FloatSlider(value=1.0, min=-4.0, max=4.0, step=0.1, description='Y Upper (d):')

col1 = VBox([mu_x_s, sigma_x_s, mu_y_s, sigma_y_s, rho_s])
col2 = VBox([a_s, b_s, c_s, d_s])
ui = HBox([col1, col2])

out = widgets.interactive_output(plot_multivariate_prob, {
    'mu_x': mu_x_s, 'mu_y': mu_y_s, 'sigma_x': sigma_x_s, 'sigma_y': sigma_y_s,
    'rho': rho_s, 'a': a_s, 'b': b_s, 'c': c_s, 'd': d_s
})

display(ui, out)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive, VBox, HBox

def simulate_gbm(mu, sigma, S0, paths, T, steps):
    """
    Simulates and visualizes Geometric Brownian Motion (GBM) paths
    to demonstrate deterministic drift vs. stochastic diffusion.
    """
    np.random.seed(42) # Seeded for consistent structural exploration
    dt = T / steps
    t = np.linspace(0, T, steps + 1)

    # 1. THE LANGUAGE & CHANGE: Calculate the random increments (dW)
    # dW ~ N(0, sqrt(dt))
    dW = np.random.normal(0, np.sqrt(dt), size=(steps, paths))

    # Accumulate the increments over time using Itô's Lemma analytic solution:
    # S(t) = S0 * exp((mu - 0.5 * sigma^2)*t + sigma * W(t))
    W = np.vstack([np.zeros(paths), np.cumsum(dW, axis=0)])

    # Calculate the stochastic matrix of paths
    time_grid = t[:, np.newaxis]
    S = S0 * np.exp((mu - 0.5 * sigma**2) * time_grid + sigma * W)

    # Calculate the purely deterministic trend line (Zero Diffusion / No Noise)
    # S_det(t) = S0 * exp(mu * t)
    S_deterministic = S0 * np.exp(mu * t)

    # 2. VISUALIZATION: Dual-panel plot (Paths on Left, Distribution on Right)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [3, 1]}, sharey=True)

    # Left Panel: Simulated Stochastic Trajectories
    ax1.plot(t, S, color='#1f77b4', alpha=0.25, lw=1)
    # Highlight a single path to show an individual realization clearly
    ax1.plot(t, S[:, 0], color='#1f77b4', lw=2, label='Individual Random Path')
    # Plot the underlying deterministic base flow
    ax1.plot(t, S_deterministic, color='#d62728', lw=3, linestyle='--', label=f'Deterministic Flow (Drift $\mu$={mu})')

    ax1.set_title(f'Stochastic Calculus Path Flow: Geometric Brownian Motion', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Time ($t$)', fontsize=11)
    ax1.set_ylabel('Asset Value ($X_t$)', fontsize=11)
    ax1.grid(True, linestyle=':', alpha=0.6)
    ax1.legend(loc='upper left')
    ax1.set_xlim(0, T)

    # Right Panel: Accumulation Histogram (Terminal Log-Normal Distribution)
    terminal_prices = S[-1, :]
    ax2.hist(terminal_prices, bins=30, orientation='horizontal', color='#2ca02c', alpha=0.6, density=True)

    # Calculate expected mean outcome at terminal time step T
    mean_terminal = np.mean(terminal_prices)
    ax2.axhline(mean_terminal, color='purple', linestyle=':', lw=2, label=f'Sample Mean: {mean_terminal:.1f}')

    ax2.set_title('Terminal Probability\nDistribution Density', fontsize=11, fontweight='bold')
    ax2.set_xlabel('Density ($f(x)$)', fontsize=11)
    ax2.grid(True, linestyle=':', alpha=0.6)
    ax2.legend(loc='upper right')

    # Equalize Y-axis ranges across panels smoothly
    max_y = max(np.max(S), np.max(S_deterministic)) * 1.05
    ax1.set_ylim(0, max_y)

    plt.tight_layout()
    plt.show()

# --- CONFIGURING INTERACTIVE CONTROL SLIDERS ---
mu_slider = widgets.FloatSlider(value=0.1, min=-0.5, max=1.0, step=0.05, description='Drift (μ):')
sigma_slider = widgets.FloatSlider(value=0.2, min=0.01, max=0.8, step=0.05, description='Volatility (σ):')
s0_slider = widgets.FloatSlider(value=100.0, min=10.0, max=200.0, step=5.0, description='Start Value (S0):')
paths_slider = widgets.IntSlider(value=100, min=10, max=500, step=10, description='Paths Num:')

# Grouping layout interface blocks together neatly
ui = HBox([VBox([mu_slider, sigma_slider]), VBox([s0_slider, paths_slider])])

out = widgets.interactive_output(simulate_gbm, {
    'mu': mu_slider, 'sigma': sigma_slider, 'S0': s0_slider, 'paths': paths_slider,
    'T': widgets.fixed(1.0), 'steps': widgets.fixed(252) # Fixed to 1 year of 252 trading days
})

display(ui, out)

1. Dynamic Spaces & Manipulating Uncertainty"these spaces aren't fixed—they're dynamic, governed by stochastic processes where uncertainty itself becomes an object to be manipulated."The standard calculus version: You have a fixed graph on a piece of paper. You draw fixed rectangles under a static curve to find the area. The paper doesn't move.The stochastic version: Imagine drawing your rectangles on a sheet of rubber, and someone is shaking the sheet up and down while you try to measure it. The space itself is dynamic because time ($dt$) and randomness ($dW_t$) are happening simultaneously.Manipulating Uncertainty: In standard math, your rectangle's width is a fixed step: $\Delta x$. In stochastic calculus, the width of your step is a random variable. Because we know the statistical rules of how that step shakes, we can mathematically add, subtract, and multiply the "shakiness" itself. Uncertainty isn't a mistake; it is the actual width of our dynamic rectangles.

2. Evolving Distributions Along Trajectories"understanding how probability distributions evolve along trajectories"The standard calculus version: If you push a particle along a line, its trajectory is a single, predictable path.The stochastic version: Because the path is shaking, a single particle splits into an infinite number of possible "ghost paths."The Rectangle Intuition: Instead of tracking a single rectangle moving along a line, imagine you drop a droplet of food coloring into water. At time $t=0$, you have one dense rectangle of color. As time moves forward ($dt$), that single rectangle splits and diffuses outward into a wide, shallow wave of millions of tiny rectangles. Tracking how that cloud of rectangles spreads out over time is what it means for a "distribution to evolve along a trajectory."

3. Density Gradients vs. Drift and Diffusion"how gradients in probability density relate to drift and diffusion"This is the core mechanic of how our tiny rectangles move. Every random process is a tug-of-war between two forces:Drift (The Conveyor Belt): This is a smooth, predictable force pushing your rectangles in a specific direction (e.g., gravity pulling a particle down, or a stock generally growing over time).Diffusion (The Shaker): This is the random force slicing and scattering your rectangles evenly in all directions.Gradients in Probability Density: A "gradient" just means a slope—a difference in height between adjacent rectangles. If one area has a tall stack of probability rectangles and the neighboring area is empty, Diffusion acts like wind, naturally blowing the rectangles down the slope to fill the empty space. The physical movement (drift and diffusion) perfectly dictates how the mathematical slopes (gradients) flatten out.

4. Tangent Spaces, Manifolds, and Risk Curvature"how the tangent spaces of manifolds capture the curvature of financial risk."A Manifold: This is just a fancy mathematical word for a curved surface (like a sphere, or a complex multidimensional hill representing financial markets).The Tangent Space: In basic calculus, if you have a curved line, you zoom in close until it looks perfectly flat, and you draw a straight tangent line to measure the slope ($dy/dx$). If you are on a 3D curved hill, you zoom in until the hill looks flat, and you place a flat sheet of cardboard against it. That flat cardboard is the tangent space.Capturing Risk Curvature: In finance, assets don't move in straight lines; they bend based on complex market forces. If you are standing on a highly curved hill of risk, your flat "tangent cardboard" will only accurately predict the hill for a split second before the hill curves away from it.By looking at how drastically your flat tangent spaces must tilt and warp as you move from one point to the next, you are measuring the curvature (or acceleration) of risk. It’s the exact equivalent of noticing that your tiny calculus rectangles have to rapidly change their heights to keep up with a steeply accelerating curve.